In [0]:
%run ../gold/00_gold_helpers

In [0]:
logger = get_logger("gold_store_sales")

try:

    logger.info("Starting Gold Store Sales transformation")

    # ---------------------------------------------------------
    # Read Stores
    # ---------------------------------------------------------

    logger.info("Reading Silver Stores table")

    df1 = read_table("stores_clean")

    logger.info("Silver Stores table read successfully")

    logger.info(
        "Removing technical columns from Stores"
    )

    df1 = df1.drop(
        "_ingestion_timestamp",
        "_source_file"
    )

    logger.info("Store technical columns removed")

    display(df1)

    # ---------------------------------------------------------
    # Read Sales
    # ---------------------------------------------------------

    logger.info("Reading Silver Sales table")

    df2 = read_table("sales_clean")

    logger.info("Silver Sales table read successfully")

    logger.info(
        "Removing technical columns from Sales"
    )

    df2 = df2.drop(
        "_ingestion_timestamp",
        "_source_file"
    )

    logger.info("Sales technical columns removed")

    display(df2)

    # ---------------------------------------------------------
    # Join Stores and Sales
    # ---------------------------------------------------------

    logger.info(
        "Joining Sales with Stores using store_id"
    )

    df = df2.join(
        df1,
        "store_id",
        how="right"
    )

    logger.info(
        "Sales and Stores join completed"
    )

    display(df)

    # ---------------------------------------------------------
    # Store Sales Aggregation
    # ---------------------------------------------------------

    logger.info(
        "Aggregating sales by store_id, store_name, city and region"
    )

    df_agg = (
        df
        .groupBy(
            "store_id",
            "store_name",
            "city",
            "region"
        )
        .agg(
            sum(
                col("quantity")
            ).alias("units_sold"),

            sum(
                col("quantity") * col("unit_price")
            ).alias("revenue")
        )
    )

    logger.info(
        "Store sales aggregation completed"
    )

    # ---------------------------------------------------------
    # Create Schema
    # ---------------------------------------------------------

    logger.info(
        f"Creating schema if it does not exist: "
        f"{catalog_name}.{schema_name}"
    )

    spark.sql(
        f"""
        CREATE SCHEMA IF NOT EXISTS
        {catalog_name}.{schema_name}
        """
    )

    logger.info(
        f"Schema ready: {catalog_name}.{schema_name}"
    )

    # ---------------------------------------------------------
    # Save Gold Table
    # ---------------------------------------------------------

    logger.info(
        "Saving Gold Store Sales Summary table"
    )

    save_table(
        df_agg,
        "stores_sales_summary"
    )

    logger.info(
        "Gold Store Sales Summary table saved successfully"
    )

    logger.info(
        "Gold Store Sales transformation completed successfully"
    )

except Exception:

    logger.exception(
        "Gold Store Sales transformation failed"
    )

    raise